In [1]:
pip install langchain langchain-community langchain-core pydantic ollama chromadb sentence-transformers


  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/536.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/536.7 kB ? eta -:--:--
   ------------------- -------------------- 262.1/536.7 kB ? eta -:--:--
   ---------------------------------------- 536.7/536.7 kB 2.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [ollama]
  Attempting uninstall: huggingface-hub
   ---------------------------------------- 0/2 [ollama]
    Found existing installation: huggingface-hub 0.36.0
   ---------------------------------------- 0/2 [ollama]
    Uninstalling huggingface-hub-0.36.0:
   ---------------------------------------- 0/2 [ollama]
      Successfully uninstalled huggingface-hub-0.36.0
   ---------------------------------------- 0/2 [ollama]
   -------------------- ------------------- 1/2 [huggingface-hub]
   -------------------- ------------------- 1/2 [hu

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 1.2.0 requires huggingface-hub<1.0.0,>=0.33.4, but you have huggingface-hub 1.3.7 which is incompatible.


# Part1

# Task1

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama

llm = Ollama(model="gemma3")

prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer the following question clearly and concisely:

Question: {question}
""",
    input_variables=["question"],
)

# Inject user input dynamically
chain = prompt | llm


In [6]:
questions = [
    "What is LangChain?",
    "Explain embeddings in simple terms",
    "What is a vector database?"
]

for q in questions:
    print(chain.invoke({"question": q}))


LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs) like GPT-3. 

**Essentially, it provides tools and abstractions to:**

*   **Connect LLMs to external data sources:**  Allows LLMs to access and use information beyond their initial training.
*   **Chain together LLM calls:** Enables you to create complex workflows with multiple LLM interactions.
*   **Build sophisticated applications:** Facilitates the creation of chatbots, question-answering systems, and more.


**In short, it makes it easier to build intelligent applications using LLMs.**

You can learn more here: [https://www.langchain.com/](https://www.langchain.com/)
Okay, here’s an explanation of embeddings in simple terms:

**Embeddings are like numerical representations of words, phrases, or even entire concepts.** 

Instead of just treating words as strings of letters, embeddings map them to points in a multi-dimensional space.  Words with similar meanings are

# Task2

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

chat_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are an expert AI tutor."),
    HumanMessage(content="{question}")
])

chat_chain = chat_prompt | llm


In [7]:
chat_chain.invoke({"question": "Explain RAG in 2 sentences"})


"Okay, I'm ready! Please ask your question. I'll do my best to explain it clearly and help you understand the concepts involved. Let's learn together! 😊 \n\nJust paste your question here."

# Part2

# Task3

In [10]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class Answer(BaseModel):
    answer: str = Field(description="Final answer")
    confidence: float = Field(description="Confidence score between 0 and 1")
    source: str = Field(description="Source of information")

parser = PydanticOutputParser(pydantic_object=Answer)


In [11]:
prompt = PromptTemplate(
    template="""
Answer the question below.
{format_instructions}

Question: {question}
""",
    input_variables=["question"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser


In [12]:
chain.invoke({"question": "What is LCEL?"})


Answer(answer='LCEL (Linear Cognition Engine Library) is an open-source, extensible framework for building cognitive architectures and intelligent agents. It provides a set of tools and components for modeling human cognition, including memory, perception, action, and learning.', confidence=0.95, source='Wikipedia')

# Task4

In [16]:
safe_chain = chain.with_retry(
    stop_after_attempt=3
)

safe_chain.invoke({"question": "Explain vector embeddings"})



Answer(answer="Vector embeddings are a way to represent words, phrases, or even entire documents as numerical vectors in a multi-dimensional space.  The key idea is that items with similar meanings are located close together in this space, while dissimilar items are farther apart.  Here's a breakdown of how they work:", confidence=0.95, source='Various sources including research papers on word embeddings (Word2Vec, GloVe, FastText) and articles explaining the concept of semantic similarity in machine learning.')

# Part3

# Task5

In [17]:
simple_chain = prompt | llm
simple_chain.invoke({"question": "What is a transformer model?"})


'```json\n{\n  "answer": "A transformer model is a type of neural network architecture that relies on a self-attention mechanism to process sequential data, such as text or audio. Unlike recurrent neural networks, transformers can process the entire input sequence simultaneously, allowing them to capture long-range dependencies more effectively. They\'ve become extremely popular in natural language processing tasks like translation, text generation, and question answering.",\n  "confidence": 0.95,\n  "source": "https://en.wikipedia.org/wiki/Transformer_model"\n}\n```'

# Task6

In [18]:
def is_factual(question: str) -> bool:
    keywords = ["who", "when", "where", "capital", "year"]
    return any(k in question.lower() for k in keywords)

def router(inputs):
    q = inputs["question"]
    if is_factual(q):
        return retriever_chain.invoke(inputs)
    return llm.invoke(q)


# Task7

In [19]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel({
    "answer": llm,
    "summary": llm,
    "follow_up": llm
})

parallel_chain.invoke("Explain LangChain")


{'answer': 'Okay, let\'s break down LangChain. It\'s a rapidly evolving framework designed to make building applications powered by large language models (LLMs) like GPT-3, GPT-4, Claude, and others *much* easier. Think of it as a toolkit and a way to orchestrate the interactions between LLMs and other data sources.\n\n**Here\'s a breakdown of the key concepts:**\n\n**1. The Problem LangChain Solves:**\n\n* **LLMs are Powerful, But Limited:** LLMs are fantastic at generating text, translating languages, and answering questions. However, they have several limitations:\n    * **Context Window:** They can only "remember" a limited amount of information at a time (the context window).  Long conversations or complex tasks can quickly overwhelm them.\n    * **Data Access:** LLMs don\'t inherently *know* things beyond what they were trained on. To make them truly useful for specific applications, you often need to connect them to external data sources.\n    * **Complexity of Building Chains:*

# Part4

# Task8

In [20]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"question": RunnablePassthrough()}
    | prompt
    | llm
)

chain.invoke("What is LCEL?")


'```json\n{\n  "answer": "LCEL stands for the Lightweight Chemical Exchange Library. It is a software library developed by the U.S. Department of Energy\'s National Renewable Energy Laboratory (NREL) for simulating chemical reaction kinetics and transport phenomena, particularly for heterogeneous catalysis.",\n  "confidence": 0.95,\n  "source": "https://www.nrel.gov/research/software/lcel/"\n}\n```'

# Task9

In [21]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_texts(
    texts=[
        "LangChain enables chaining LLM components.",
        "LCEL is a declarative chain syntax."
    ],
    embedding=embeddings
)

retriever = vectorstore.as_retriever()


C:\Windows\Temp\ipykernel_26932\1181442416.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 348.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | PromptTemplate.from_template(
        "Context: {context}\nQuestion: {question}"
    )
    | llm
)


In [23]:
rag_chain.invoke("What is LCEL?")


'According to the provided documents, LCEL is a declarative chain syntax. It’s also a feature that LangChain enables, allowing for the chaining of LLM components.'

# Task10

1️ Why structured output matters

Ensures reliability

Machine-readable responses

Prevents hallucinated formats

2️ LCEL advantages

Declarative & readable

Easy composition

Native parallelism

3️ Parallel vs Conditional

Parallel: independent outputs (summary + answer)

Conditional: decision-based routing (RAG vs LLM)